In [1]:
import re
import pandas as pd
import torch

dialogues = [('How do I change my email?', 'You can change your email in account settings.'), ('How can I reset my password?', 'Open the login page and use the password reset link.'), ('I forgot my password.', 'Use the password recovery form to create a new password.'), ('How do I contact support?', 'You can contact support through the help center.'), ('Where can I see my orders?', 'Open your profile and select the orders section.'), ('How do I cancel an order?', 'Open the order details and choose cancel.'), ('Can I change my delivery address?', 'Yes, change it before the order is shipped.'), ('When will my order arrive?', 'The estimated delivery date is shown in order details.'), ('How do I track my package?', 'Use the tracking number from your shipping confirmation.'), ('My package is late.', 'Please check tracking information or contact support.'), ('Can I get a refund?', 'Eligible purchases can be refunded according to the policy.'), ('How long does a refund take?', 'Refunds usually take several business days.'), ('How do I update my phone number?', 'Go to account settings and edit your phone number.'), ('How do I delete my account?', 'Open account settings and select account deletion.'), ('How do I create an account?', 'Click sign up and enter the required information.'), ('I cannot log in.', 'Check your email and password and try again.'), ('How do I change my username?', 'Edit your username in profile settings.'), ('Where are my account settings?', 'Click your profile icon and open settings.'), ('How do I add a payment method?', 'Open payment settings and choose add payment method.'), ('Can I remove my card?', 'Yes, remove a saved card from payment settings.'), ('Why was my payment declined?', 'Check your payment details or try another method.'), ('What payment methods do you accept?', 'We accept the methods listed at checkout.'), ('How do I change the language?', 'Select your preferred language in application settings.'), ('How do I enable notifications?', 'Open notification settings and turn them on.'), ('How do I turn off notifications?', 'Open notification settings and turn them off.'), ('Where can I find the help center?', 'You can open the help center from the support menu.'), ('How do I report a problem?', 'Send a report through the support form.'), ('Can I change my order?', 'You may change an order before it is processed.'), ('How do I check my order status?', 'Open your orders page to see the current status.'), ('Thank you for your help.', 'You are welcome. We are happy to help.')]
df = pd.DataFrame(dialogues, columns=["question", "answer"])
print("Кількість пар:", len(df))
display(df.head())

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z0-9]+|[?.!,]", text)

df["question_tokens"] = df["question"].apply(tokenize)
df["answer_tokens"] = df["answer"].apply(tokenize)

print("\nПриклад токенізації:")
print(df.loc[0, "question_tokens"])
print(df.loc[0, "answer_tokens"])

PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

all_tokens = []
for col in ["question_tokens", "answer_tokens"]:
    for tokens in df[col]:
        all_tokens.extend(tokens)

vocab = [PAD, UNK, SOS, EOS] + sorted(set(all_tokens))
token_to_idx = {token: i for i, token in enumerate(vocab)}
idx_to_token = {i: token for token, i in token_to_idx.items()}

print("\nРозмір словника:", len(vocab))

def encode(tokens):
    return [token_to_idx[SOS]] + [token_to_idx.get(t, token_to_idx[UNK]) for t in tokens] + [token_to_idx[EOS]]

df["question_ids"] = df["question_tokens"].apply(encode)
df["answer_ids"] = df["answer_tokens"].apply(encode)

PAD_IDX = token_to_idx[PAD]

def pad(seq, length):
    return seq[:length] + [PAD_IDX] * max(0, length - len(seq))

max_q = max(map(len, df["question_ids"]))
max_a = max(map(len, df["answer_ids"]))

df["question_padded"] = df["question_ids"].apply(lambda x: pad(x, max_q))
df["answer_padded"] = df["answer_ids"].apply(lambda x: pad(x, max_a))

X = torch.tensor(df["question_padded"].tolist(), dtype=torch.long)
Y = torch.tensor(df["answer_padded"].tolist(), dtype=torch.long)

print("\nРезультат:")
print("Пар діалогів:", len(df))
print("Словник:", len(vocab))
print("Макс. довжина запиту:", max_q)
print("Макс. довжина відповіді:", max_a)
print("X shape:", X.shape)
print("Y shape:", Y.shape)

display(df[["question", "answer", "question_padded", "answer_padded"]].head(10))

df[["question", "answer", "question_padded", "answer_padded"]].to_csv(
    "seq2seq_dialogues_prepared.csv", index=False, encoding="utf-8-sig"
)
print("\nЗбережено: seq2seq_dialogues_prepared.csv")


ModuleNotFoundError: No module named 'torch'